# LoanWatch (Google Colab 版)

名古屋グランパスのレンタル移籍選手について、最新試合の出場状況を一覧にします。
Windows 版の `LoanWatch.exe` と同じ処理を、ブラウザだけで実行できます。

## 使い方

上から順にセルを実行してください（各セルの左の ▶ をクリック）。

| 手順 | セル | 所要時間の目安 |
| :---- | :---- | :---- |
| 1 | 準備（パッケージのインストール） | 2〜4分 |
| 2 | ファイルのアップロード | すぐ |
| 3 | 動作確認 | 30秒 |
| 4 | 実行 | 選手数×20秒ほど |
| 5 | 結果の表示 | すぐ |
| 6 | ダウンロード | すぐ |

## 必要なファイル

手順2で次の4つをアップロードします。Windows 版と同じフォルダに入っているものです。

- `LoanWatchMain.py` … 本体
- `version.py` … バージョン表示
- `u21_clubs.py` … U-21 Ｊリーグのクラブ一覧
- `players.json` … 選手設定

## Colab で使うときの注意

- ランタイムが切断されると、アップロードしたファイルは消えます。もう一度手順1からやり直してください。
- Colab は Google のデータセンターから通信するため、取得先のサイトからボット扱いされ、
  自宅の回線より取得に失敗しやすいことがあります。失敗した選手は `備考` 列に理由が出ます。
- GPU は不要です。ランタイムのタイプは `CPU` のままで構いません。


In [ ]:
#@title 手順1: 準備（パッケージのインストール） { display-mode: "form" }
# 2〜4分かかります。最後に「準備完了」と出れば成功です。
import importlib.util

print("playwright をインストールしています...")
!pip install -q playwright

# pandas や numpy を入れ直すとランタイムの再起動を求められるため、
# Colab に最初から入っているものには触らず、足りないものだけ入れる
NEEDED = {"requests": "requests", "bs4": "beautifulsoup4", "lxml": "lxml",
          "html5lib": "html5lib", "pandas": "pandas", "openpyxl": "openpyxl"}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("不足しているパッケージを追加します:", " ".join(missing))
    !pip install -q {" ".join(missing)}
else:
    print("必要なパッケージはすべて入っています")

print("\nブラウザ(Chromium)をインストールしています...")
!playwright install --with-deps chromium

print("\n準備完了")

In [ ]:
#@title 手順2: ファイルのアップロード { display-mode: "form" }
# 下の4つのファイルを選んでください（まとめて選択できます）。
import pathlib
from google.colab import files

REQUIRED = ("LoanWatchMain.py", "version.py", "u21_clubs.py", "players.json")
missing = [n for n in REQUIRED if not pathlib.Path(n).is_file()]

if missing:
    print("次のファイルを選んでください:", " / ".join(missing))
    files.upload()
else:
    print("アップロード済みです。入れ替えたい場合は、左のフォルダアイコンから削除して再実行してください。")

still_missing = [n for n in REQUIRED if not pathlib.Path(n).is_file()]
if still_missing:
    print("\n[エラー] まだ足りないファイルがあります:", " / ".join(still_missing))
else:
    import json
    players = json.loads(pathlib.Path("players.json").read_text(encoding="utf-8"))["players"]
    print(f"\n選手設定を読み込みました: {len(players)}人")
    for p in players:
        extra = " + BeSoccer" if p.get("besoccer") else ""
        print(f"  - {p['name_ja']} ({p['club']}) [{p['source']}{extra}]")

In [ ]:
#@title 手順3: 動作確認 { display-mode: "form" }
# ブラウザが起動できるかだけを確かめます。PLAYWRIGHT_SMOKE_TEST_OK と出れば正常です。
!python LoanWatchMain.py --playwright-smoke-test

In [ ]:
#@title 手順4: 実行 { display-mode: "form" }
#@markdown 対象シーズン（西暦）
YEAR = 2026 #@param {type:"integer"}
#@markdown 特定の選手だけ実行する場合は名前を入れます（空欄なら全員）
ONLY = "" #@param {type:"string"}
#@markdown BeSoccer からのリザーブチームの行を出さない
NO_BESOCCER = False #@param {type:"boolean"}
#@markdown U-21 ＪリーグのクラブID（別シートに最新試合のメンバーを出す。空欄なら出さない）
U21_CLUB = "33186" #@param {type:"string"}
#@markdown 出力ファイル名（拡張子なし）
OUT = "loan_players" #@param {type:"string"}

import shlex, subprocess, os

cmd = ["python", "LoanWatchMain.py", "--year", str(YEAR), "--out", OUT]
if ONLY.strip():
    cmd += ["--only", *ONLY.split()]
if NO_BESOCCER:
    cmd += ["--no-besoccer"]
cmd += ["--u21-club", U21_CLUB.strip()] if U21_CLUB.strip() else ["--no-u21-team"]

# Colab の時刻は UTC のため、日付や「経過」が日本時間とずれないようにする
env = dict(os.environ, TZ="Asia/Tokyo")
print("実行:", " ".join(shlex.quote(c) for c in cmd), "\n")
done = subprocess.run(cmd, env=env)
if done.returncode:
    print("\n[注意] 取得できなかった選手がいます。手順5の 備考 列で理由を確認してください。")

In [ ]:
#@title 手順5: 結果の表示 { display-mode: "form" }
import pathlib
import pandas as pd

csv = pathlib.Path(f"{OUT}.csv")
if not csv.is_file():
    print(f"{csv} がありません。手順4を先に実行してください。")
else:
    pd.set_option("display.max_colwidth", 40)
    pd.set_option("display.width", 200)
    # 数値に見える列を勝手に float にされると 0 が 0.0 になるので、文字列のまま読む
    df = pd.read_csv(csv, dtype=str).fillna("")
    display(df.drop(columns=["URL"]))

    u21_csv = pathlib.Path(f"{OUT}_u21.csv")
    if u21_csv.is_file():
        print("\nU-21 Ｊリーグ 最新試合のメンバー:")
        display(pd.read_csv(u21_csv, dtype=str).fillna("").drop(columns=["URL"]))

    # 取得できなかった行だけを抜き出す
    ng = df[df["ステータス"] == "試合なし"]
    if len(ng):
        print(f"\n取得できなかった行が {len(ng)} 件あります。理由は次のとおりです。")
        for _, r in ng.iterrows():
            print(f"  - {r['選手']} / {r['クラブ']}: {r['備考']}")

In [ ]:
#@title 手順6: ダウンロード { display-mode: "form" }
import pathlib
from google.colab import files

# U-21 のCSVは、U-21を取得したときだけ作られる
for name in (f"{OUT}.xlsx", f"{OUT}.csv", f"{OUT}_u21.csv"):
    if pathlib.Path(name).is_file():
        files.download(name)

## 困ったときは

### `PLAYWRIGHT_SMOKE_TEST_OK` が出ない

手順1をもう一度実行してください。それでも駄目な場合は、メニューの
`ランタイム` → `ランタイムを接続解除して削除` でランタイムを作り直し、手順1からやり直します。

### 特定の選手だけ `試合なし` になる

`備考` 列に理由が入っています。よくあるものは次のとおりです。

| 備考の内容 | 意味 |
| :---- | :---- |
| `HTTP 406 ボット判定でブロック` / `ボット確認ページ(CAPTCHA)` | 取得先にボット扱いされています。時間をおいて再実行してください |
| `2026年の試合が0件` | そのシーズンの試合がまだないか、選手設定の値が違います |
| `2026年の試合がまだない(中止・延期など)` | シーズン開幕前などで、正常な状態です |

Colab は Google のデータセンターから通信するため、自宅の回線よりブロックされやすい傾向があります。
Windows 版の `LoanWatch.exe` で実行すると通ることがあります。

### 選手を追加・変更したい

`players.json` を書き換えてアップロードし直してください。Windows 版の
`選手設定を編集` 画面で編集したものをそのまま使えます。

### 毎回アップロードするのが面倒

Google ドライブに必要なファイルを置いておき、
手順2の代わりに次を実行するとコピーできます（`MyDrive/LoanWatch/` に置いた場合）。

```python
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/LoanWatch/LoanWatchMain.py .
!cp /content/drive/MyDrive/LoanWatch/version.py .
!cp /content/drive/MyDrive/LoanWatch/u21_clubs.py .
!cp /content/drive/MyDrive/LoanWatch/players.json .
```
